# Automotive Hood Stress Surrogate  
## From a global regressor to a regime-aware Mixture-of-Experts model

## Objective

Stress prediction proved substantially more difficult than deformation prediction.

The purpose of this notebook is not only to present the final model, but to document the **engineering reasoning that guided the model-development process**.

The progression was:

1. Global PointNet-style regression
2. Error analysis by stress range
3. High-stress-tail diagnosis
4. Design-family and activation-pattern investigation
5. Geometry-regime analysis
6. Parameter-regime analysis
7. Mixture-of-Experts model
8. Final challenge experiment using PointNet++ + multitask learning

### Final selected stress surrogate

On the held-out target-stratified test set:

- **R² = 0.9055**
- **MAE = 8.47 MPa**
- **RMSE = 14.05 MPa**

The model also substantially reduced high-stress underprediction compared with the earlier global regressors.

> **Validation note:**  
> This is a representative target-stratified holdout benchmark. It is not a topology-disjoint deployment test.

## 1. Why stress is harder than deformation

Global deformation is strongly related to overall stiffness and geometry.

Peak stress is different.

Stress can be dominated by:
- local geometric details,
- load-path changes,
- structural discontinuities,
- specific design regimes,
- parameter interactions,
- localized concentration effects.

Therefore, a model can predict average behavior reasonably well while still failing badly on the most important high-stress designs.

For engineering use, that tail behavior matters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. First global stress surrogate

The initial PointNet-style stress model used:
- 1,024 centered point-cloud points
- preserved physical scale
- 54 design parameters

The first global model achieved approximately:

- **MAE ≈ 9.55 MPa**
- **RMSE ≈ 16.69 MPa**
- **R² ≈ 0.879**

At first glance, this looked respectable.

However, global R² alone was not sufficient to judge engineering usefulness.

## 3. Engineering judgement: inspect the stress tail

Instead of stopping at global metrics, the predictions were broken down into stress ranges.

This was an important engineering decision because the most safety-relevant errors occur at the upper end of the stress distribution.

The stress bands used were:

- <100 MPa
- 100–125 MPa
- 125–150 MPa
- 150–175 MPa
- 175–200 MPa
- 200–250 MPa
- 250–300 MPa
- ≥300 MPa

In [ ]:
stress_bands = [
    "<100", "100-125", "125-150", "150-175",
    "175-200", "200-250", "250-300", ">=300"
]
stress_bands

### What the band analysis revealed

The global model was acceptable in the dense middle of the distribution but deteriorated sharply in the upper tail.

Representative results:

- **250–300 MPa:** MAE ≈ 21.4 MPa
- **≥300 MPa:** MAE ≈ 84.2 MPa
- **≥300 MPa bias:** approximately **−73 MPa**

This was not a small statistical imperfection.

It was a systematic engineering problem:

> The model was strongly underpredicting the designs where stress was highest.

## 4. Do not remove the high-stress cases

One possible ML response would be to classify these cases as outliers and remove them.

That was deliberately rejected.

> **Engineering judgement:**  
> High-stress cases are physically important in a CAE surrogate. Unless a run is proven to be numerically corrupted, removing the difficult high-stress samples would make the model appear better while reducing its engineering value.

The goal therefore became:

**Improve the model's handling of the high-stress regime rather than remove it.**

## 5. Diagnostic study of the high-stress designs

There were only about **45 designs above 300 MPa**, making this a sparse regime.

Compared with the rest of the dataset, these designs showed distinct characteristics.

Typical differences included:

- higher deformation
- lower mass
- different parameter activation patterns
- stronger presence/absence of specific design variables

The high-stress designs were therefore not simply random noise.

In [ ]:
high_stress_summary = pd.DataFrame({
    "Quantity": ["Stress", "Deformation", "Mass"],
    "High-stress mean": [340.88, 17.94, 11.896],
    "Other designs mean": [157.28, 10.094, 13.84]
})
high_stress_summary

## 6. Activation patterns suggested design regimes

Several parameters had very different activation frequencies in the high-stress designs.

For example, parameters such as P9, P19, P39, and P40 were more frequently active, while some others were absent.

This suggested that the upper tail was associated with particular design configurations.

However, activation pattern alone was not enough.

Within one exact activation family, stress still ranged approximately from **68 MPa to 352 MPa**.

### Engineering interpretation

The same nominal design family can contain very different structural behavior.

Therefore, the model needed to distinguish not only:

**which parameters are active**

but also:

**what geometry regime the design belongs to and what parameter magnitudes occur within that regime.**

## 7. Geometry regime inside an exact design family

Within one exact activation family of approximately 81 designs:

- geometry descriptors alone separated low-stress and moderate/high-stress regimes very effectively,
- a simple two-cluster geometry split clearly isolated one low-stress cluster from one moderate/high-stress cluster.

This was a key turning point.

> **Engineering judgement:**  
> The stress response was behaving hierarchically. A single smooth global mapping was forcing physically different regimes into the same regression function.

## 8. Parameter magnitude matters inside the geometry regime

After isolating the moderate/high geometry regime, the 54 parameters became much more informative.

Within that regime, parameter-based classification of ≥300 MPa cases achieved very strong discrimination.

P40, in particular, showed a strong relationship with stress.

But there was an important caution.

The same P40 value could still correspond to very different stresses.

### Engineering interpretation

P40 was treated as a **regime indicator**, not a causal predictor.

The observed hierarchy was approximately:

**Topology / family → geometry regime → parameter magnitude & interactions → stress**

## 9. Why a Mixture-of-Experts architecture was chosen

The diagnostic studies suggested that stress was better represented as several related sub-problems rather than one homogeneous regression problem.

That motivated a **3-expert Mixture-of-Experts (MoE)** model.

The architecture contains:

### Shared representation
- point-cloud encoder
- 54-parameter encoder
- engineered geometry-descriptor encoder

### Three regression experts
Each expert learns a different part of the stress mapping.

### Gating network
The gate assigns a soft probability to each expert.

Final prediction:

**Stress = w₁·Expert₁ + w₂·Expert₂ + w₃·Expert₃**

The gate was not explicitly told which stress band each expert should represent.

## 10. Final multimodal MoE architecture

Inputs:

- **1,024 centered 3D points**
- **54 design parameters**
- **24 engineered geometry descriptors**

Point-cloud scale was preserved by dividing globally by 1000 after centering.

The model used:
- PointNet-style global encoder
- max + mean pooling
- separate parameter encoder
- separate geometry-descriptor encoder
- shared fusion layers
- three stress experts
- softmax gate

In [ ]:
import torch
import torch.nn as nn

class StressExpert(nn.Module):
    def __init__(self, in_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


class Gate(nn.Module):
    def __init__(self, in_dim=128, n_experts=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_experts)
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=1)

The training loss contained:
- stress mean-squared error,
- a very small gate-balance penalty.

No stress-band labels were used to supervise the gate.

## 11. Target-stratified train / validation / test split

Because the high-stress tail is sparse, a purely random split can easily place too few high-stress samples in one subset.

A target-stratified split was therefore used to preserve the stress distribution.

Approximate split:

- Train: **7,944**
- Validation: **993**
- Test: **994**

The ≥300 MPa cases were distributed approximately as:

- Train: 36
- Validation: 5
- Test: 4

### Important interpretation

This improves consistency of model comparison, but it is still a **retrospective representative benchmark**, not a deployment-like unseen-topology test.

## 12. Final stress MoE result

The final 3-expert model achieved:

- **MAE = 8.4655 MPa**
- **RMSE = 14.0492 MPa**
- **R² = 0.9055**

This was the best global stress result obtained in the model-development study.

In [ ]:
stress_results = pd.DataFrame({
    "Model": [
        "PointNet + 54 params",
        "Weighted-MSE PointNet",
        "PointNet++",
        "Final 3-expert MoE"
    ],
    "R2": [0.8791, 0.879, 0.8568, 0.9055]
})
stress_results

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(stress_results["Model"], stress_results["R2"])
plt.ylabel("R²")
plt.ylim(0.80, 0.93)
plt.xticks(rotation=30, ha="right")
plt.title("Stress Model Progression")
plt.tight_layout()
plt.show()

## 13. Tail improvement

The most important improvement was not only the global R².

The high-stress tail improved materially.

### Earlier PointNet

For **≥250 MPa**:
- MAE ≈ 32.31 MPa
- RMSE ≈ 55.54 MPa
- Bias ≈ −30.32 MPa

### Final MoE

For **≥250 MPa**:
- **MAE ≈ 18.62 MPa**
- **RMSE ≈ 31.37 MPa**
- **Bias ≈ −16.83 MPa**

The underprediction problem was not eliminated, but it was substantially reduced.

## 14. Expert specialization emerged automatically

One of the most interesting findings was that the three experts spontaneously specialized.

Typical behavior:

- lower-stress designs were routed mostly to one expert,
- intermediate regimes shifted toward another,
- higher-stress designs increasingly favored a different expert.

The gate was not directly trained with stress-band classes.

> **Engineering interpretation:**  
> This supports the earlier hypothesis that the stress response contains multiple latent structural regimes.

## 15. Final challenge experiment: PointNet++ + multitask learning + MoE

To ensure that the final model was not simply winning because the architecture comparison was too weak, a more sophisticated challenge model was tested.

The challenge model used:

- PointNet++ local neighborhood learning
- 2,048 points
- 54 design parameters
- three experts for stress
- auxiliary deformation prediction
- auxiliary mass prediction
- target-stratified 80/10/10 split
- high-stress cases retained

This was intentionally a strong comparison.

### Challenge-model result

The PointNet++ multitask MoE achieved approximately:

- **Stress R² = 0.8795**
- **Stress MAE = 9.27 MPa**
- **Stress RMSE = 15.86 MPa**

It performed very well on the auxiliary targets:

- Deformation R² ≈ **0.959**
- Mass R² ≈ **0.984**

But stress performance was worse than the selected MoE.

### High-stress behavior of the challenge model

For stress ≥250 MPa, the PointNet++ multitask model had approximately:

- **MAE = 30.46 MPa**
- **RMSE = 42.20 MPa**
- **Bias = −30.46 MPa**

For the ≥300 MPa band:

- **MAE ≈ 75.22 MPa**
- strong systematic underprediction remained.

One case was approximately:

- Actual: **331 MPa**
- Predicted: **204 MPa**

This confirmed that a more complex local geometric architecture did not solve the stress-regime problem.

## 16. Why the final MoE was selected

The final model was selected because it provided the best balance of:

- global stress accuracy,
- reduced high-stress-tail error,
- interpretable expert specialization,
- physically meaningful input representation,
- consistent performance on the representative holdout.

The key lesson was:

> **Model sophistication alone did not solve the problem. The decisive improvement came from diagnosing the engineering regimes in the data and choosing an architecture consistent with those regimes.**

## 17. Engineering decisions that changed the model

### Decision 1 — Preserve physical scale
Unit-sphere normalization was rejected because it removed absolute dimensions that can affect structural response.

### Decision 2 — Keep the high-stress cases
They were not discarded as outliers because they are precisely the cases a CAE surrogate must handle responsibly.

### Decision 3 — Evaluate by stress band
Global R² hid severe high-stress underprediction.

### Decision 4 — Investigate design families
Activation patterns and geometry families revealed that the data were not homogeneous.

### Decision 5 — Treat parameter importance carefully
Parameters such as P40 were interpreted as regime indicators rather than claimed as direct causal drivers.

### Decision 6 — Use regime-aware modeling
The hierarchical behavior motivated a Mixture-of-Experts architecture.

### Decision 7 — Stop when the evidence was sufficient
A stronger PointNet++ multitask challenge model was tested and rejected because it did not improve the primary stress objective.

## 18. Limitations

This study still has important limitations:

1. The final benchmark is target-stratified rather than topology-disjoint.
2. Very-high-stress cases remain sparse.
3. The point cloud represents geometry but not full solver-state information.
4. Material, thickness, connection, or loading information may be encoded only indirectly through available parameters.
5. A surrogate should support engineering screening, not replace CAE validation for final sign-off.

A future deployment-oriented study should evaluate:
- unseen topology families,
- leave-family-out validation,
- uncertainty estimation,
- active learning around the high-stress regime.

## 19. Final project takeaway

The stress study demonstrates the value of combining engineering judgement with machine learning.

The progression was not:

**bigger network → better model**

It was:

**observe failure → diagnose the physical regime → redesign the model → validate the improvement**

The final stress surrogate therefore represents not only a predictive model, but an engineering reasoning process.